## Welcome to the Second Lab - Week 1, Day 3

Today we will work with lots of models! This is a way to get comfortable with APIs.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important point - please read</h2>
            <span style="color:#ff7800;">The way I collaborate with you may be different to other courses you've taken. I prefer not to type code while you watch. Rather, I execute Jupyter Labs, like this, and give you an intuition for what's going on. My suggestion is that you carefully execute this yourself, <b>after</b> watching the lecture. Add print statements to understand what's going on, and then come up with your own variations. See Q37 in the <a href="https://edwarddonner.com/avatar?q=37">FAQ</a> for how to set up a separate project for your work.<br/><br/>If you have time, I'd love it if you submit a PR for changes in the community_contributions folder - instructions in the resources. Also, if you have a Github account, use this to showcase your variations. Not only is this essential practice, but it demonstrates your skills to others, including perhaps future clients or employers...<br/>And if you post about it on LinkedIn and tag me, then I'll weigh in to amplify your achievement. If you see other students posting, please give them your encouragement too.
            </span>
        </td>
    </tr>
</table>

In [1]:
# Start with imports - ask the Cursor Agent to explain any package that you don't know

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [2]:
# Always remember to do this!
load_dotenv(override=True)

True

In [3]:
# Print the key prefixes to help with any debugging

openai_api_key = os.getenv('OPENAI_API_KEY')

google_api_key = os.getenv('GOOGLE_API_KEY')

openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}") 
else:
    print("OpenRouter API Key not set (and this is optional)")


OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AQ
OpenRouter API Key exists and begins sk-or-


In [4]:
request = """
Please come up with a challenging, nuanced question with a succinct answer,
that I can ask a number of LLMs to evaluate their intelligence.
Not a mathematical puzzle, but more of a thought-provoking question that requires intelligent insight.
Include in your question that the answer must be short.
"""
request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]

In [5]:
messages

[{'role': 'user',
  'content': '\nPlease come up with a challenging, nuanced question with a succinct answer,\nthat I can ask a number of LLMs to evaluate their intelligence.\nNot a mathematical puzzle, but more of a thought-provoking question that requires intelligent insight.\nInclude in your question that the answer must be short.\nAnswer only with the question, no explanation.'}]

In [6]:
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
question = response.choices[0].message.content
display(Markdown(question))

You are given a perfectly rational agent that always chooses the option maximizing expected utility, but its utility function can change after every decision in ways the agent cannot predict. What single principle, if any, should it still follow to remain meaningfully “rational,” and why? Answer in one short sentence.

## Calling LLMs from multple providers

We are about to call LLMs from many other providers.
They all provide API endpoints that are compatible with OpenAI, as explained in Guide 9 in the guides folder.
So we can simply use these endpoints as if we are using OpenAI.

Please note:

I'm going to use lots of LLMs from different providers, but you don't need to! This is only to show their abilities.

In [7]:
# OpenAI Compatible URLs

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

In [8]:
# OpenAI client libraries with the right base_url and key
# If this surprises you, please see Guide 9 in the Guides folder!
gemini = OpenAI(api_key=google_api_key, base_url=GEMINI_BASE_URL)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=OPENROUTER_BASE_URL)
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

In [9]:
competitors = []
answers = []
messages = [{"role": "user", "content": question}]

In [10]:
def record(model_name, answer):
    competitors.append(model_name)
    answers.append(answer)
    display(Markdown(answer))

In [11]:
# The API we know well
# Reasoning effort can be none, low, medium, high, or xhigh

model_name = "gpt-5.4-nano"

response = openai.chat.completions.create(model=model_name, messages=messages, reasoning_effort="none")
answer = response.choices[0].message.content

record(model_name, answer)

It should follow **robust expected-utility optimality**—i.e., always choose the action that maximizes expected utility under the only information it can stably rely on (its current beliefs), because if the utility can change unpredictably the only rational constraint left is to act optimally with respect to the present, not to the future.

In [12]:
model_name = "gemini-3.1-flash-lite"

response = gemini.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

record(model_name, answer)

The agent should consistently act to maximize its current utility, as the unpredictable nature of future preferences renders any attempt at intertemporal consistency futile and logically ill-defined.

In [23]:
model_name = "openai/gpt-4o-mini"

response = openrouter.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

record(model_name, answer)


The agent should consistently choose the action that maximizes its expected utility based on its current utility function, as this maintains the rationality of decision-making in the face of uncertainty.

## For the next cell, we will use Ollama

Ollama runs a local web service that gives an OpenAI compatible endpoint,  
and runs models locally using high performance C++ code.

If you don't have Ollama, install it here by visiting https://ollama.com then pressing Download and following the instructions.

After it's installed, you should be able to visit here: http://localhost:11434 and see the message "Ollama is running"

You might need to restart Cursor (and maybe reboot). Then open a Terminal (control+\`) and run `ollama serve`

Useful Ollama commands (run these in the terminal, or with an exclamation mark in this notebook):

`ollama pull <model_name>` downloads a model locally  
`ollama ls` lists all the models you've downloaded  
`ollama rm <model_name>` deletes the specified model from your downloads

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Super important - ignore me at your peril!</h2>
            <span style="color:#ff7800;">Many models on Ollama are FAR too large for your home computer. Be sure to browse the models on the Ollama website. Look to use models that are size 3GB or less unless you know better; llama3.2 is a great first choice. Don't pick models that end in :cloud; that's something different (a cloud inference service, like Groq).
            </span>
        </td>
    </tr>
</table>

In [24]:
!ollama pull llama3.2:1b

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling 74701a8c35f6: 100% ▕██████████████████▏ 1.3 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 4f659a1e86d7: 100% ▕██████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
success 


In [25]:
import requests
requests.get('http://localhost:11434').content

b'Ollama is running'

In [29]:
import requests
models = requests.get('http://localhost:11434/v1/models').json()
for model in models.get("data"):
    print(model)

{'id': 'llama3.2:1b', 'object': 'model', 'created': 1785566369, 'owned_by': 'library'}


In [30]:
model_name = "llama3.2:1b"

response = ollama.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

record(model_name, answer)

The single principle that the rational agent should still follow is Bayes' theorem, as it allows for the consideration of uncertain outcomes when selecting options based on expected utility.

In [32]:
# It's nice to know how to use "zip"
for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}")


Competitor: gpt-5.4-nano

It should follow **robust expected-utility optimality**—i.e., always choose the action that maximizes expected utility under the only information it can stably rely on (its current beliefs), because if the utility can change unpredictably the only rational constraint left is to act optimally with respect to the present, not to the future.
Competitor: gemini-3.1-flash-lite

The agent should consistently act to maximize its current utility, as the unpredictable nature of future preferences renders any attempt at intertemporal consistency futile and logically ill-defined.
Competitor: openai/gpt-4o-mini

The agent should consistently apply the principle of maximizing expected utility based on its current utility function for each decision, as this approach remains the best way to align its choices with its evolving preferences and objectives.
Competitor: openai/gpt-4o-mini

The agent should consistently choose the action that maximizes its expected utility based o

In [33]:
# Let's bring this together - note the use of "enumerate"

together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

In [34]:
print(together)

# Response from competitor 1

It should follow **robust expected-utility optimality**—i.e., always choose the action that maximizes expected utility under the only information it can stably rely on (its current beliefs), because if the utility can change unpredictably the only rational constraint left is to act optimally with respect to the present, not to the future.

# Response from competitor 2

The agent should consistently act to maximize its current utility, as the unpredictable nature of future preferences renders any attempt at intertemporal consistency futile and logically ill-defined.

# Response from competitor 3

The agent should consistently apply the principle of maximizing expected utility based on its current utility function for each decision, as this approach remains the best way to align its choices with its evolving preferences and objectives.

# Response from competitor 4

The agent should consistently choose the action that maximizes its expected utility based on 

In [35]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""


In [36]:
print(judge)

You are judging a competition between 5 competitors.
Each model has been given this question:

You are given a perfectly rational agent that always chooses the option maximizing expected utility, but its utility function can change after every decision in ways the agent cannot predict. What single principle, if any, should it still follow to remain meaningfully “rational,” and why? Answer in one short sentence.

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}

Here are the responses from each competitor:

# Response from competitor 1

It should follow **robust expected-utility optimality**—i.e., always choose the action that maximizes expected utility under the only information it can stably rely on (its current beliefs), because if the utility can c

In [37]:
judge_messages = [{"role": "user", "content": judge}]

## And now for Grok!

Branded as "The most truth-seeking large language model in the world".. so let's use it as our LLM as a judge

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Which pattern(s) did this use? Try updating this to add another Agentic design pattern.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">These kinds of patterns - to send a task to multiple models, and evaluate results,
            are common where you need to improve the quality of your LLM response. This approach can be universally applied
            to business projects where accuracy is critical.
            </span>
        </td>
    </tr>
</table>